In [386]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

In [387]:
#carichiamo il csv con i soggetti di Wikidata
df_wikidata = pd.read_csv("wd_dataset.csv", encoding="utf-8")

df_wikidata.head(3)


,entity,label,aliases,description,category
0,http://www.wikidata.org/entity/Q304227,Aba,NaN,ninfa naiade della mitologia greca,greek_deity
1,http://www.wikidata.org/entity/Q279782,Abarbarea,NaN,naiade della mitologia greca,greek_deity
2,http://www.wikidata.org/entity/Q2370052,Acanto,NaN,"personaggio della mitologia greca, probabilmente una ninfa amata da Apollo",greek_deity


In [388]:
#estraggo tutte le label di wikidata 
labels = (
    df_wikidata["label"]
    .dropna() #tolgo i valori nulli
    .str.strip() #tolgo gli spazi bianchi all'inizio e alla fine
    .tolist() #converto in lista

)

#estraggo gli aliases di wikidataseparando quelli multipli
aliases = []
for alias_str in df_wikidata["aliases"].dropna(): #per ogni stringa di alias che non è nulla
    alias_list = [alias.strip() for alias in alias_str.split(",")] #separo gli alias con la virgola, tolgo gli spazi e metto tutto in minuscolo
    aliases.extend(alias_list) #aggiungo alla lista totale degli alias

#unisco tutto togliendo i duplicati
myth_terms = list(set(labels + aliases)) 
len(myth_terms)

2973

In [389]:
#ENDPOINT SPARLQ DI ARCO 
arco_endpoint = "https://dati.cultura.gov.it/sparql"

sparql_arco = SPARQLWrapper(arco_endpoint)
sparql_arco.setReturnFormat(JSON)

In [390]:
# NUMERO TOTALE DEI BENI STORICO ARTISTICI - senza duplicati 
query1 = """
PREFIX arco: <https://w3id.org/arco/ontology/arco/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#> 

SELECT (COUNT(DISTINCT ?item) AS ?totalItems)
WHERE {
  ?item rdf:type <https://w3id.org/arco/ontology/arco/HistoricOrArtisticProperty> .
}
"""


sparql_arco.setQuery(query1)
results = sparql_arco.query().convert()

for result in results["results"]["bindings"]:
    print(f"Total Historic or Artistic Properties: {result['totalItems']['value']}")

Total Historic or Artistic Properties: 2258640


In [391]:
#query pr estrarre subject e label dei beni storico artistici in batch (altrimenti esplode)
# Iterare su tutti i 2.258.640 beni con batch di 20000 (altrimenti esplode)

rows = []
batch_size = 20000
total_items = 2258640

for offset in range(0, total_items, batch_size):
    print(f"Processando batch: offset={offset}, items processati={len(rows)}")
    
    query2 = f"""
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX a-cd: <https://w3id.org/arco/ontology/context-description/>

SELECT ?item ?subject ?label
WHERE {{

  ?item rdf:type <https://w3id.org/arco/ontology/arco/HistoricOrArtisticProperty> .
  
  OPTIONAL {{ ?item rdfs:label ?label . }}

  OPTIONAL {{ ?item a-cd:subject ?subject . }}

}}
LIMIT {batch_size}
OFFSET {offset}
"""
    
    sparql_arco.setQuery(query2)
    results = sparql_arco.query().convert()
    
    # estrai i dati da questo batch
    for r in results["results"]["bindings"]:
        rows.append({
            "item": r["item"]["value"] if "item" in r else None,
            "label": r["label"]["value"] if "label" in r else None,
            "subject": r["subject"]["value"] if "subject" in r else None
        })
    
    # se il batch è vuoto, ferma il loop
    if len(results["results"]["bindings"]) < batch_size:
        print(f"Batch incompleto ({len(results['results']['bindings'])} items), fine loop")
        break


Processando batch: offset=0, items processati=0
Processando batch: offset=20000, items processati=20000
Processando batch: offset=40000, items processati=40000
Processando batch: offset=60000, items processati=60000
Processando batch: offset=80000, items processati=80000
Processando batch: offset=100000, items processati=100000
Processando batch: offset=120000, items processati=120000
Processando batch: offset=140000, items processati=140000
Processando batch: offset=160000, items processati=160000
Processando batch: offset=180000, items processati=180000
Processando batch: offset=200000, items processati=200000
Processando batch: offset=220000, items processati=220000
Processando batch: offset=240000, items processati=240000
Processando batch: offset=260000, items processati=260000
Processando batch: offset=280000, items processati=280000
Processando batch: offset=300000, items processati=300000
Processando batch: offset=320000, items processati=320000
Processando batch: offset=340000

In [392]:

# creiamo il dataframe consolidato
df_arco = pd.DataFrame(rows)
df_arco = df_arco.drop_duplicates(subset="item")

print(f"Righe totali in df_arco : {len(df_arco)}")
df_arco[["item", "label", "subject"]].head(1)

Righe totali in df_arco : 1120195


,item,label,subject
0,https://w3id.org/arco/resource/HistoricOrArtisticProperty/0500704280,"Arlecchino primordiale. Fine del cinquecento. Martinelli, figurino per costume di scena, maschile (disegno) by Fo, Dario (XX)","figurino per costume di scena, maschile"


In [400]:
import re

# 1. filtro base (toglie null)
df_arco = df_arco[df_arco["subject"].notna()].copy()

terms = [
    re.escape(t)
    for t in myth_terms
    if isinstance(t, str)
    and t.strip() != ""
    and t[0].isupper()
]

# regex unica case-sensitive (matching esatto)
pattern = r'(?<![A-Za-zÀ-ÿ])(' + '|'.join(terms) + r')(?![A-Za-zÀ-ÿ-])'
regex = re.compile(pattern)


#TERMINI DA ACCETTARE SOLO SE COMPAIONO INSIEME A QUESTE PAROLE []
context_rules = {
    "Ascanio" : ["fuoco", "cervo"],"Callisto": ["Arcade", "Diana"],"Concordia": ["Allegoria", "Innocenza", "Giustizia"],
    "Cornacchia": ["metamorfosi"],"Egeo": ["Teseo"],"Elena": ["ratto"],"Ettore": ["morte", "Protesilao", "Achille", "Andromaca", "Paride"],
    "Europa": ["ratto", "Toro", "sibilla", "mito","allegoria","ancelle","leone"],
    "Grazie" : ["tre", "danzanti"],"Ida" : ["Linceo"],"Ippolito":["episodi", "caduta", "morte"],
    "Nilo":["personificazione","allegoria"],"Roma":["dea"],"Silvano":["Dio"],"Sonno":["allegoria"],
    "Vesta":["Giove"],"Vittoria":["allegoria"]
}


# TERMINI DA ESCLUDERE SE COMPAIONO CON DETERMINATE PAROLE
exclusion_rules = {
    "Abbondanza": ["Santa", "Sant","san","papa"],"Amore": ["Dio", "Madonna", "castello", "fontana","antica"],"Andromeda":["galassia"],"Apollo":["sala","tempio"],
    "Atlante":["Scienze","turisti"],"Carità": ["Santa", "Sant", "Maria"],
    "Carita": ["Santa", "Sant", "Maria"],"Diana" : ["Beata", "duca","scenografia","tempio","conte","san"],
    "Dioniso": ["stemma","stemmi", "san", "santi", "s."],"Dionisio":["Ritratto","san","barletta","beati","vescovo","Santi","stemma","papa","padre","beato"],
    "Elio" : ["ponte","ritratto","paesaggio", "veduta"],"Enea": ["Beato", "discepolo","stemma", "papa", "silvio"],
    "Ercole": ["Ricotti", "Dandini", "Comini", "Farnese","cardinale", "tempio", "stemma", "ritratto", "Roccotti", "colonne", "porto", "basilica", "santa"],
    "Ermes" : ["ritratto"],"Esculapio": ["tempio"],
    "Fede" : ["santi", "sant", "san", "dottrina", "biblici", "Francia", "cattolica", "madonna", "evangelisti", "religione", "dio", "angeli", "angioletti","santa", "papa", "santo", "cristiana", "chiesa", "miracolo", "misericordia","virtù", "ritratto", "stemma"],
    "Flora" :["ritratto", "sante", "circo", "ruota", "iside", "madonna", "tempio"],"Galatea":["ritratto"],
    "Giasone":["san","ritratto"],"Giove":["san","tempio"],
    "Giunone":["scenografia", "tempio"],"Giustizia" : ["porta", "palazzo", "misericordia", "san", "santa", "papa", "imperatore", "papale","Virtù", "apostoli","papa","Giglio"],
    "Gorgone" : ["santa"],"greco":["testo","teatro"],"Io": ["ritratti "],"Ippolito":["Nievo"],"Lavinia":["ritratto", "autoritratto"],
    "Leandro" : ["autoritratto","ritratto","stemma","San", "busto", "torre"],"Marte":["tempio","sala","ultore"],
    "Medea":["Colleoni"],"Mercurio":["tempio","san","assiso"],"Mirra":["San"],"Minerva":["autoritratto","chiesa","tempio"],
    "Narciso":["san"],"Nereo":["ritratto","san","santi","SS."],"Nettuno":["veduta"],"Oceano":["Atlantico"],
    "Oreste":["Ritratto","autoritratto"],"Pan":["zucchero","veduta"],"Patroclo":["san"],"Pirro":["Stipiciano"],"Pluto":["Topolino","Disney"],
    "Polidoro":["ritratto","Vicolo"],"Polissena":["Ritratto","santa"],"Priamo":["san","santo","Sulis","Fumagalli"],"Scilla":["topografica","veduta","paesaggio"],
    "Telemaco":["stemma","ritratto"],"Teseo":["Spirito"],"Tolomeo":["San","Santi","Ritratto","beato"],"Urano":["Monte"],
    "Venere":["Giorgio","Castello","modella","tempio"],"Verità":["stemma","Pierantonio","ritratto"],"Vittorie":["reggitarga"],
    "Speranza":["ritratto","santa","chiesa","papa","san","santo","madonna"],"Scilla":["ritratto"],"Serapide":["tempio"],"Achille":["stemma","ritratto","duomo","san"],
    "Adone":["sant"],"Cibele":["tempio"],"Cirene":["simone","palazzo"],"Teseo":["tempio","ritratto"],"Titano":["san"],"Ulisse":["ritratto","Dini","cardinalizia"],"Zeus":["tempio"]

}


def validate_matches(row):

    subject = row["subject"]

    valid_matches = []

    for match in row["matched_term"]:

        # PRIMO CHECK: controlla se il termine è nelle exclusion_rules
        if match in exclusion_rules:
            exclusion_keywords = exclusion_rules[match]
            # se almeno una parola di esclusione è presente, scarta il termine
            if any(re.search(r'\b' + re.escape(k) + r'\b', subject, re.IGNORECASE) for k in exclusion_keywords):
                continue  
        
        # SECONDO CHECK: controlla context_rules (inclusione condizionata)
        # se il termine non ha regole → tienilo
        if match not in context_rules:
            valid_matches.append(match)
            continue

        # controlla parole contestuali con word boundaries (parole intere)
        keywords = context_rules[match]

        if any(re.search(r'\b' + re.escape(k) + r'\b', subject, re.IGNORECASE) for k in keywords):
            valid_matches.append(match)

    return valid_matches


df_arco["matched_term"] = df_arco["subject"].str.findall(regex)
# matching veloce
df_arco["matched_term"] = df_arco.apply(validate_matches, axis=1)

# solo match
df_matches = df_arco[df_arco["matched_term"].str.len() > 0]



In [401]:
# Verifica i risultati
print(f"Totale beni Arco: {len(df_arco)}")
print(f"Beni con match mitologico: {len(df_matches)}")

Totale beni Arco: 522845
Beni con match mitologico: 10633


In [402]:
#cosa facciamo con Lucifero? 
#cosa facciamo con Notte? 
#cosa facciamo con la Fede? 

In [403]:
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", None) 

start = 10500
end = 10700



df_matches_sorted = df_matches.sort_values("matched_term")

df_matches_sorted[["item","label", "subject", "matched_term"]].iloc[start:end]


,item,label,subject,matched_term
397464,https://w3id.org/arco/resource/HistoricOrArtisticProperty/1500860624-1,allegoria della Vittoria come donna vestita all'antica (monumento ai caduti - a lapide) - bottega italiana (sec. XX),allegoria della Vittoria come donna vestita all'antica,[Vittoria]
861388,https://w3id.org/arco/resource/HistoricOrArtisticProperty/1500817226,allegoria della Vittoria (monumento ai caduti) - bottega Italia centro-meridionale (secondo quarto sec. XX),allegoria della Vittoria,[Vittoria]
806138,https://w3id.org/arco/resource/HistoricOrArtisticProperty/0900559021,allegoria della Vittoria come aquila (lapide commemorativa ai caduti) - bottega toscana (sec. XX),allegoria della Vittoria come aquila,[Vittoria]
1072216,https://w3id.org/arco/resource/HistoricOrArtisticProperty/0800577674,allegoria della Vittoria come aquila (monumento ai caduti - a colonna) by Ditta Michieli (sec. XX),allegoria della Vittoria come aquila,[Vittoria]
1072032,https://w3id.org/arco/resource/HistoricOrArtisticProperty/0800577519,allegoria della Vittoria come aquila (monumento ai caduti - ad obelisco) di Ditta Davide Venturi & Figlio (sec. XX),allegoria della Vittoria come aquila,[Vittoria]
378018,https://w3id.org/arco/resource/HistoricOrArtisticProperty/1200218434-1,"allegoria della Vittoria come aquila (rilievo, serie) di Cambellotti Duilio (sec. XX)",allegoria della Vittoria come aquila,[Vittoria]
861322,https://w3id.org/arco/resource/HistoricOrArtisticProperty/1500817199,allegoria della Vittoria come aquila (monumento ai caduti - a lapide) by Palmieri Gabriele e Figli (sec. XX),allegoria della Vittoria come aquila,[Vittoria]
829008,https://w3id.org/arco/resource/HistoricOrArtisticProperty/1300282825,"Allegoria della Vittoria (monumento ai caduti - a cippo, opera isolata) - ambito abruzzese (seconda metà XX)",Allegoria della Vittoria,[Vittoria]
829024,https://w3id.org/arco/resource/HistoricOrArtisticProperty/1300282927,Allegoria della Vittoria come aquila (monumento ai caduti - a cippo) - ambito abruzzese (prima metà XX),Allegoria della Vittoria come aquila,[Vittoria]
861324,https://w3id.org/arco/resource/HistoricOrArtisticProperty/1500817201-0,allegoria della Vittoria come donna vestita all'antica (monumento ai caduti - a colonna) di Jerace Francesco (sec. XX),allegoria della Vittoria come donna vestita all'antica,[Vittoria]


In [404]:
#esportiamo i risultati in un file csv
df_matches_sorted.to_csv("arco_myth_matches.csv", index=False, encoding="utf-8")


